# PPO Training (Colab + Drive)

This notebook mounts Drive (optional), runs PPO training, and plots saved metrics.

In [ ]:
from pathlib import Path
import torch

try:
    import torchmetrics  # noqa: F401
    TORCHMETRICS_OK = True
    TORCHMETRICS_ERR = ''
except Exception as exc:
    TORCHMETRICS_OK = False
    TORCHMETRICS_ERR = str(exc)

IN_COLAB = True

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/rl_quantum_circuit_routing')
else:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / 'data'
RUN_ROOT = PROJECT_ROOT / 'runs'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('torchmetrics:', 'OK' if TORCHMETRICS_OK else f'MISSING ({TORCHMETRICS_ERR})')

In [ ]:
!pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

In [ ]:
RUN_NAME = 'ppo_colab_run1'
CMD = (
    f"python {PROJECT_ROOT / 'main.py'} "
    f"--in-colab "
    f"--project-root {PROJECT_ROOT} "
    f"--run-name {RUN_NAME} "
    f"--device auto "
    f"--topologies heavy_hex_19,grid_3x3,linear_5 "
    f"--total-timesteps 200000"
)
print(CMD)
!{CMD}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = RUN_ROOT / RUN_NAME / 'metrics.csv'
df = pd.read_csv(metrics_path)
display(df.tail())

fig, axs = plt.subplots(1, 3, figsize=(18, 4))
axs[0].plot(df['global_step'], df['mean_step_reward'])
axs[0].set_title('Mean Step Reward')
axs[0].set_xlabel('global_step')

axs[1].plot(df['global_step'], df['mean_episode_return'])
axs[1].set_title('Mean Episode Return')
axs[1].set_xlabel('global_step')

axs[2].plot(df['global_step'], df['value_loss'], label='value_loss')
axs[2].plot(df['global_step'], df['policy_loss'], label='policy_loss')
axs[2].set_title('Losses')
axs[2].set_xlabel('global_step')
axs[2].legend()
plt.tight_layout()

print('Checkpoints:')
for p in sorted((RUN_ROOT / RUN_NAME).glob('*.pt')):
    print('-', p.name)